# Notebook 03: Dataset Integration

## Purpose
Merge child, maternal, and household datasets to build the modeling dataset.

## Objectives
1. Load cleaned child dataset  
2. Load IR and HR datasets  
3. Merge datasets using DHS identifiers  
4. Select core modeling variables  

## Output
data/processed/model_dataset.parquet

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
# Define paths
PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

In [3]:
# Load cleaned KR dataset
df_kr = pd.read_parquet(DATA_INTERIM / "kr_clean.parquet")

# Load IR and HR datasets
df_ir = pd.read_stata(DATA_RAW / "MWIR81FL.dta")
df_hr = pd.read_stata(DATA_RAW / "MWHR81FL.dta")

print(df_kr.shape)
print(df_ir.shape)
print(df_hr.shape)

(5415, 1210)
(21587, 5879)
(23095, 3067)


## Merge datasets

This section links child records with maternal and household data using DHS identifiers.

In [4]:
# Merge child with mother data
df_merged = df_kr.merge(
    df_ir,
    on=["v001", "v002", "v003"],
    how="left"
)

print(df_merged.shape)

(5415, 7086)


In [5]:
# Merge with household data
df_merged = df_merged.merge(
    df_hr,
    left_on=["v001", "v002"],
    right_on=["hv001", "hv002"],
    how="left"
)

print(df_merged.shape)

(5415, 10153)


## Select core modeling variables

This section extracts key variables for the first modeling dataset.

In [7]:
[col for col in df_merged.columns if "v012" in col]
[col for col in df_merged.columns if "v106" in col]
[col for col in df_merged.columns if "v190" in col]
[col for col in df_merged.columns if "v025" in col]

['v025_x', 'v025_y', 'hv025']

In [8]:
# Select core features
features = [
    "v001", "v002",
    "hw70", "stunted",
    "v012_x",   # mother age
    "v106_x",   # education
    "v190_x",   # wealth
    "v025_x",   # urban or rural
    "hv009"     # household size
]

df_model = df_merged[features].copy()

print(df_model.shape)
df_model.head()

(5415, 9)


,v001,v002,hw70,stunted,v012_x,v106_x,v190_x,v025_x,hv009
0,1,9,-0.84,0,27,primary,poorer,rural,4
1,1,24,-2.52,1,40,primary,richer,rural,7
2,1,24,-1.98,0,40,primary,richer,rural,7
3,1,39,-0.67,0,24,primary,richer,rural,3
4,1,69,-2.04,1,29,primary,richer,rural,9


In [9]:
df_model.isna().mean()

v001       0.0
v002       0.0
hw70       0.0
stunted    0.0
v012_x     0.0
v106_x     0.0
v190_x     0.0
v025_x     0.0
hv009      0.0
dtype: float64

In [11]:
# Convert categorical columns before saving
for col in df_model.select_dtypes(["category"]).columns:
    df_model[col] = df_model[col].astype(str)

# Save modeling dataset for downstream use
df_model.to_parquet(DATA_PROCESSED / "model_dataset.parquet", index=False)

In [12]:
(DATA_PROCESSED / "model_dataset.parquet").exists()

True

## Summary

The child, maternal, and household datasets were successfully merged.

Key variables were selected to form the initial modeling dataset.  
The dataset contains no missing values in the selected features.  
This dataset is ready for exploratory analysis and modeling.

The processed dataset has been saved for use in the next stage.